# Wazuh — odtwarzalny trening końcowego GRU

**Cel:** od plików AIT-ADS `*_wazuh.json` i `labels.csv` do wytrenowanego modelu sekwencyjnego.

Ten notatnik jest **wersją uporządkowaną na podstawie projektu badawczego**, a nie historią wszystkich eksperymentów. Nie zawiera porównań RF/LSTM, prób doboru progu, diagnostyki Dovecot ani testów HTTP. Do zwykłego **uruchomienia API nie trzeba wykonywać tego notatnika** — wystarczą `api.py`, `wazuh_predictor.py` i gotowy checkpoint.

### Założenia i ograniczenia
- Tylko alerty Wazuh, bez Suricaty i AMiner; filtr odpowiada końcowemu eksperymentowi.
- Etykiety `labels.csv` są **etykietami czasowymi**, a nie pewną klasyfikacją każdego alertu.
- Scenariusze rozwojowe: FOX, RussellMitchell, Shaw, Wardbeck, Wheeler, Wilson, Santos. Harrison służył do jednorazowej oceny końcowej; nie jest używany w treningu.
- Model przewiduje `background` i dziewięć faz ataku. `service_stop` występuje w definicji zadania, ale nie ma przykładów aktywnych okien — model nie potrafi go przewidywać.
- Przykładowe wartości IP, nazwy agentów, bezwzględny czas i etykieta fazy nie są cechami wejścia modelu. Czas służy do agregacji i przypisania etykiet.
- **Nie nadpisujemy istniejącego modelu końcowego.** Nowe wagi zapisujemy pod nazwą `wazuh_gru_reproduced.pt`.

**Wymagania:** Python, pandas, NumPy, PyTorch, ipykernel. Uruchamiaj z katalogu głównego projektu, gdzie istnieje `data/ait_ads/`. Brak surowego zbioru AIT-ADS uniemożliwia odtworzenie treningu. Kod został uporządkowany na podstawie przesłanego notatnika, ale w tym środowisku **nie był wykonywany na pełnych danych**.

## 1. Ścieżki, etykiety i filtr źródeł

Weryfikujemy obecność danych oraz wczytujemy przedziały `[start,end)` z `labels.csv`. Przypadki z Suricaty/AMiner są wyłączane. W końcowym eksperymencie do budowy danych używaliśmy filtracji `location` zawierającej `suricata`; przed rozpoczęciem nowego treningu zachowujemy tę samą definicję, zamiast bez uzasadnienia zmieniać dane.

In [8]:
from pathlib import Path
from collections import defaultdict
import csv
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path("data/ait_ads")
SCENARIOS = ["fox", "harrison", "russellmitchell", "santos", "shaw", "wardbeck", "wheeler", "wilson"]
TRAIN_SCENARIOS = ["fox", "russellmitchell", "shaw", "wardbeck", "wheeler", "wilson", "santos"]
TEST_SCENARIO = "harrison"

assert (DATA_DIR / "labels.csv").is_file(), "Brak labels.csv; sprawdź katalog roboczy"
for scenario in SCENARIOS:
    assert (DATA_DIR / f"{scenario}_wazuh.json").is_file(), f"Brak pliku: {scenario}_wazuh.json"

def load_intervals(path):
    result = defaultdict(list)
    with path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            scenario = row["scenario"].strip().lower()
            start, end = float(row["start"]), float(row["end"])
            assert end > start, f"Błędny przedział: {scenario}"
            result[scenario].append((start, end, row["attack"].strip()))
    for scenario in result:
        result[scenario].sort()
    return result

intervals = load_intervals(DATA_DIR / "labels.csv")
print("Scenariusze:", SCENARIOS)
print("Epizody w etykietach:", sum(len(x) for x in intervals.values()))

Scenariusze: ['fox', 'harrison', 'russellmitchell', 'santos', 'shaw', 'wardbeck', 'wheeler', 'wilson']
Epizody w etykietach: 79


## 2. Wczytanie wyłącznie Wazuh i etykietowanie czasu

Każdy alert jest osobną obserwacją wejściową do agregacji. Pozostawiamy scenariusz tylko do rozdzielenia danych i czasu; żadna z tych wartości nie trafi do GRU. Kontrolujemy sumę **1 986 905** rekordów po filtracji w oryginalnym eksperymencie. Jeżeli liczba się różni, należy sprawdzić dane i filtr, nie pomijać ostrzeżenia.

In [9]:
rows = []
for path in sorted(DATA_DIR.glob("*_wazuh.json")):
    scenario = path.stem.removesuffix("_wazuh").lower()
    with path.open(encoding="utf-8") as file:
        for line in file:
            alert = json.loads(line)
            location = str(alert.get("location") or "").lower()
            if "suricata" in location:
                continue
            rule = alert.get("rule") or {}
            decoder = alert.get("decoder") or {}
            rows.append({
                "scenario": scenario,
                "timestamp": alert.get("@timestamp"),
                "rule_id": rule.get("id"),
                "rule_level": rule.get("level"),
                "decoder_name": decoder.get("name"),
            })

df_features = pd.DataFrame(rows)
assert len(df_features) == 1_986_905, (len(df_features), "Niezgodna liczba alertów")
df_features["event_time"] = pd.to_datetime(df_features["timestamp"], utc=True)
times = ((df_features["event_time"] - pd.Timestamp("1970-01-01", tz="UTC"))
         / pd.Timedelta(seconds=1)).to_numpy()
scenarios = df_features["scenario"].to_numpy()
labels = np.full(len(df_features), "background", dtype=object)
match_count = np.zeros(len(df_features), dtype=np.uint8)
for scenario, phase_intervals in intervals.items():
    scenario_mask = scenarios == scenario
    for start, end, phase in phase_intervals:
        in_phase = scenario_mask & (times >= start) & (times < end)
        match_count[in_phase] += 1
        labels[in_phase] = phase
assert not (match_count > 1).any(), "Nachodzące etykiety — sprawdź labels.csv"
df_features["attack_phase"] = labels
print("Alerty po filtracji:", len(df_features))
print("Rozkład faz:")
display(pd.crosstab(df_features["scenario"], df_features["attack_phase"]))

Alerty po filtracji: 1986905
Rozkład faz:


attack_phase,background,cracking,dirb,dnsteal,network_scans,privilege_escalation,reverse_shell,service_scans,webshell,wpscan
scenario,,,,,,,,,,
fox,28801,174,406075,347,123,10,0,76,0,4749
harrison,31483,349,410697,1370,72,37,3,109,15,4824
russellmitchell,15005,8,4459,383,8,6,0,79,0,3168
santos,29550,389,4459,543,24,22,3,20,9,3270
shaw,29581,239,4459,105,3,19,3,1,0,729
wardbeck,33529,384,4459,9,3,13,6,22,9,750
wheeler,47037,0,412693,175,15,9,0,69,0,6638
wilson,66435,486,423894,1101,6,22,9,75,0,3199


## 3. Okna 10 sekund i podział scenariuszy

Agregujemy alerty w okna 10-sekundowe i usuwamy okna przecinające granice faz opisanych w `labels.csv`. Każde okno ma liczniki alertów, liczbę różnych reguł, maksymalny poziom reguły i liczności 24 reguł.

**Podział końcowy:** siedem scenariuszy rozwojowych służy do treningu końcowego, a `harrison` pozostaje oddzielnym scenariuszem testowym. W tym notatniku nie tworzymy już dodatkowej walidacji `santos`, ponieważ wybór parametrów wykonano wcześniej w archiwum eksperymentów.


In [10]:
# ETAP 7 — budowa okien 10-sekundowych

import numpy as np
import pandas as pd

WINDOW_SECONDS = 10

# Minimalny zestaw kolumn, aby oszczędzać pamięć RAM.
events = df_features[
    ["scenario", "event_time", "rule_id",
     "rule_level", "attack_phase"]
].copy()

events["window_start"] = (
    events["event_time"].dt.floor(f"{WINDOW_SECONDS}s")
)

keys = ["scenario", "window_start"]

# Podstawowe statystyki każdego okna
windows = (
    events.groupby(keys)
    .agg(
        alert_count=("rule_id", "size"),
        unique_rules=("rule_id", "nunique"),
        max_rule_level=("rule_level", "max"),
        label_count=("attack_phase", "nunique"),
        attack_phase=("attack_phase", "first")
    )
)

# Liczba wystąpień każdej reguły Wazuh w oknie
rule_counts = (
    events.groupby(keys + ["rule_id"])
    .size()
    .unstack(fill_value=0)
)

rule_counts.columns = [
    f"rule_count_{rule_id}"
    for rule_id in rule_counts.columns
]

windows = windows.join(rule_counts)

# Sprawdzamy granice faz z labels.csv.
# Okno musi mieścić się w jednej fazie albo w background.
windows = windows.reset_index()

boundary_crossing = np.zeros(len(windows), dtype=bool)

for scenario, phase_intervals in intervals.items():
    mask = windows["scenario"].eq(scenario).to_numpy()
    positions = np.flatnonzero(mask)

    starts = (
        (windows.loc[mask, "window_start"]
         - pd.Timestamp("1970-01-01", tz="UTC"))
        / pd.Timedelta(seconds=1)
    ).to_numpy()

    crosses = np.zeros(len(starts), dtype=bool)

    for attack_start, attack_end, phase in phase_intervals:
        for boundary in (attack_start, attack_end):
            crosses |= (
                (starts < boundary)
                & (boundary < starts + WINDOW_SECONDS)
            )

    boundary_crossing[positions] = crosses

# Odrzucamy również okna z różnymi etykietami alertów.
invalid = (
    boundary_crossing
    | windows["label_count"].gt(1).to_numpy()
)

print("Okna z alertami:", len(windows))
print("Odrzucone okna:", int(invalid.sum()))

df_windows = (
    windows.loc[~invalid]
    .drop(columns=["label_count"])
    .reset_index(drop=True)
)

# Kontrola: sumy liczników reguł zgadzają się
# z liczbą alertów w każdym oknie.
rule_columns = [
    col for col in df_windows.columns
    if col.startswith("rule_count_")
]

assert (
    df_windows[rule_columns].sum(axis=1)
    == df_windows["alert_count"]
).all()

print("Pozostało okien:", len(df_windows))

# Rozkład etykiet na poziomie OKIEN, nie alertów
window_distribution = pd.crosstab(
    df_windows["scenario"],
    df_windows["attack_phase"]
)

print("\nROZKŁAD OKIEN WEDŁUG FAZ")
display(window_distribution)

print("\nPRZYKŁADOWE CECHY OKIEN")
display(
    df_windows[
        ["scenario", "attack_phase", "alert_count",
         "unique_rules", "max_rule_level"]
    ].head(10)
)

# Końcowy trening: 7 scenariuszy rozwojowych; Harrison wyłącznie do testu.
# Używamy nazw zdefiniowanych na początku notatnika:
# TRAIN_SCENARIOS i TEST_SCENARIO.
windows_train = df_windows.loc[
    df_windows["scenario"].isin(TRAIN_SCENARIOS)
].copy()

windows_test = df_windows.loc[
    df_windows["scenario"].eq(TEST_SCENARIO)
].copy()

assert len(windows_train) == 55_672, len(windows_train)
assert len(windows_test) == 7_560, len(windows_test)
assert set(windows_train["scenario"]).isdisjoint(
    set(windows_test["scenario"])
)

print("\nPODZIAŁ OKIEN")
print("Trening końcowy (7 scenariuszy):", len(windows_train))
print("Odłożony test Harrison (nie używany do uczenia):", len(windows_test))


Okna z alertami: 63262
Odrzucone okna: 30
Pozostało okien: 63232

ROZKŁAD OKIEN WEDŁUG FAZ


attack_phase,background,cracking,dirb,dnsteal,network_scans,privilege_escalation,reverse_shell,service_scans,webshell,wpscan
scenario,,,,,,,,,,
fox,6616,38,117,77,21,2,0,4,0,4
harrison,6998,77,157,281,15,4,1,19,2,6
russellmitchell,3605,5,1,86,5,3,0,3,0,2
santos,6389,83,2,104,4,2,0,2,3,2
shaw,7021,56,1,27,1,2,1,0,0,2
wardbeck,7498,79,2,6,1,2,1,2,2,1
wheeler,10049,0,99,43,4,2,0,1,0,3
wilson,13177,89,120,191,2,2,1,2,0,2



PRZYKŁADOWE CECHY OKIEN


,scenario,attack_phase,alert_count,unique_rules,max_rule_level
0,fox,background,3,1,3
1,fox,background,2,1,3
2,fox,background,2,1,3
3,fox,background,1,1,3
4,fox,background,1,1,3
5,fox,background,1,1,3
6,fox,background,1,1,3
7,fox,background,2,1,3
8,fox,background,2,1,3
9,fox,background,2,1,3



PODZIAŁ OKIEN
Trening końcowy (7 scenariuszy): 55672
Odłożony test Harrison (nie używany do uczenia): 7560


## 4. Sekwencja 60 sekund — sześć kroków × 27 cech

Ważna poprawka względem pierwszego prototypu: historię budujemy z **wszystkich okien źródłowych**, również tych odrzuconych jako przykłady docelowe z powodu przecięcia granic. Nie wolno zastępować ich ciszą. Liczniki przechodzą `log1p`, a `max_rule_level` pozostaje bez transformacji.

In [11]:
SEQ_LEN = 6
STEP_SECONDS = 10
sequence_features = ["alert_count", "unique_rules", "max_rule_level"] + rule_columns
count_indices = [i for i, name in enumerate(sequence_features) if name != "max_rule_level"]

all_events = df_features[["scenario", "event_time", "rule_id", "rule_level"]].copy()
all_events["window_start"] = all_events["event_time"].dt.floor("10s")
keys = ["scenario", "window_start"]
all_windows = all_events.groupby(keys).agg(
    alert_count=("rule_id", "size"),
    unique_rules=("rule_id", "nunique"),
    max_rule_level=("rule_level", "max"),
)
all_rule_counts = all_events.groupby(keys + ["rule_id"]).size().unstack(fill_value=0)
all_rule_counts.columns = [f"rule_count_{rule_id}" for rule_id in all_rule_counts.columns]
all_windows = all_windows.join(all_rule_counts).reindex(
    columns=sequence_features, fill_value=0
).astype("float32")

X_sequences = np.zeros((len(df_windows), SEQ_LEN, len(sequence_features)), dtype=np.float32)
for step in range(SEQ_LEN):
    lag = (SEQ_LEN - 1 - step) * STEP_SECONDS
    requested_index = pd.MultiIndex.from_arrays(
        [df_windows["scenario"], df_windows["window_start"] - pd.Timedelta(seconds=lag)],
        names=["scenario", "window_start"],
    )
    X_sequences[:, step, :] = all_windows.reindex(
        requested_index, fill_value=0
    ).to_numpy(dtype=np.float32)
X_sequences[:, :, count_indices] = np.log1p(X_sequences[:, :, count_indices])
assert np.isfinite(X_sequences).all()
assert X_sequences.shape == (63232, 6, 27), X_sequences.shape

# Istniejący checkpoint jest punktem odniesienia dla kolejności cech.
original_path = Path("models/wazuh_gru_multiclass.pt")
if original_path.is_file():
    original = torch.load(original_path, map_location="cpu", weights_only=True)
    assert list(original["feature_names"]) == sequence_features, "Inna kolejność cech!"

print("Sekwencje:", X_sequences.shape)
print("Cecha nr 0:", sequence_features[0])

Sekwencje: (63232, 6, 27)
Cecha nr 0: alert_count


## 5. Końcowy trening GRU na siedmiu scenariuszach

Model nie dostaje etykiet ani identyfikatorów scenariuszy jako wejścia. `Harrison` jest **wyłączony**. Trening: 12 epok, hidden size 32, Adam 0.001, CrossEntropy z maksymalną wagą klasy 10. Jest to **reprodukcja ustalonej konfiguracji**, nie nowy proces optymalizacji hiperparametrów.

In [12]:
class MulticlassGRU(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=32, num_layers=1, batch_first=True)
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        output, _ = self.gru(x)
        return self.classifier(output[:, -1, :])

final_mask = df_windows["scenario"].isin(TRAIN_SCENARIOS).to_numpy()
assert not (final_mask & df_windows["scenario"].eq(TEST_SCENARIO).to_numpy()).any()
final_phase_names = sorted(df_windows.loc[final_mask, "attack_phase"].unique())
assert len(final_phase_names) == 10 and "service_stop" not in final_phase_names
final_phase_to_id = {name: i for i, name in enumerate(final_phase_names)}
X_final = torch.from_numpy(X_sequences[final_mask].copy()).float()
y_final = torch.tensor(df_windows.loc[final_mask, "attack_phase"].map(final_phase_to_id).to_numpy(), dtype=torch.long)
assert X_final.shape[0] == 55_672, X_final.shape
counts = np.bincount(y_final.numpy(), minlength=len(final_phase_names))
weights = np.minimum(np.sqrt(counts.max() / counts.astype(float)), 10.0)

# Utrzymujemy tę samą konfigurację i kolejność losowań co w oryginalnej komórce.
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_gru = MulticlassGRU(X_sequences.shape[2], len(final_phase_names)).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
optimizer = torch.optim.Adam(final_gru.parameters(), lr=0.001)
loader = DataLoader(TensorDataset(X_final, y_final), batch_size=256, shuffle=True)
for epoch in range(1, 13):
    final_gru.train()
    loss_sum = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(final_gru(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(xb)
    print(f"Epoka {epoch:02d}: loss = {loss_sum / len(X_final):.4f}")

output_path = Path("models/wazuh_gru_reproduced.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": final_gru.cpu().state_dict(),
    "phase_names": final_phase_names,
    "feature_names": list(sequence_features),
    "sequence_length": 6,
    "window_seconds": 10,
    "hidden_size": 32,
    "epochs": 12,
    "seed": 42,
    "class_counts": counts.tolist(),
    "class_weights": weights.tolist(),
    "trained_scenarios": TRAIN_SCENARIOS,
    "unobservable_classes": ["service_stop"],
}, output_path)
print("Zapisano nowy checkpoint bez nadpisywania modelu referencyjnego:", output_path)

Epoka 01: loss = 0.8639
Epoka 02: loss = 0.5488
Epoka 03: loss = 0.5365
Epoka 04: loss = 0.5312
Epoka 05: loss = 0.5254
Epoka 06: loss = 0.5182
Epoka 07: loss = 0.5138
Epoka 08: loss = 0.5122
Epoka 09: loss = 0.5075
Epoka 10: loss = 0.5042
Epoka 11: loss = 0.5034
Epoka 12: loss = 0.5025
Zapisano nowy checkpoint bez nadpisywania modelu referencyjnego: models\wazuh_gru_reproduced.pt


Weryfikacja odtwarzalności

Potwierdzono, że trening przeprowadzony za pomocą uporządkowanego notatnika odtwarza model referencyjny. Wszystkie parametry sieci GRU były identyczne (maksymalna różnica wag: 0,0). Zgodność potwierdzono także dla architektury, kolejności klas i cech wejściowych.

## 6. Podsumowanie i interpretacja

- Zapisane wagi są wynikiem uczenia, nie sztywnej tabeli reguł.
- Oryginalny, wcześniej oceniony checkpoint znajduje się w `models/wazuh_gru_multiclass.pt`. **Nie zastępuj go** automatycznie nowo wytrenowanym modelem.
- Końcowy historyczny test na wydzielonym Harrisonie dał: **Macro F1 0,4872**, **recall wykrywania okien ataku 0,3043**, **recall poprawnego rodzaju 0,3025**; **6/9** epizodów z pełnymi oknami zostało oznaczonych jako jakikolwiek atak, **5/9** miało prawidłowy typ. Wynik dotyczy pierwotnego checkpointu, **nie nowej reprodukcji**.
- Wpływ ograniczeń źródłowej telemetrii: `cracking`, `dnsteal`, `network_scans` pozostają trudne; `service_stop` nie ma w ogóle klasy wyjściowej.
- Do serwowania predykcji używaj oddzielnego `api.py` + `wazuh_predictor.py`. Ten notatnik służy wyłącznie do odtworzenia treningu.